In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l1.6 Optimization loop

Now put it together and train a real (if tiny) language model: a **bigram**,
which predicts the next character from the current one alone. Watch the loss
fall as the optimizer does its job.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless backend (CI has no display)
import matplotlib.pyplot as plt
from torch.nn import functional as F
from pocketlm import CharTokenizer, BigramLM

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)

torch.manual_seed(0)
model = BigramLM(tok.vocab_size)
opt = torch.optim.AdamW(model.parameters(), lr=0.1)
block, batch = 32, 32
steps = 50 if os.environ.get("POCKETLM_CI") else 300
losses = []
for _ in range(steps):
    ix = torch.randint(len(data) - block - 1, (batch,))
    xb = torch.stack([data[i:i + block] for i in ix])
    yb = torch.stack([data[i + 1:i + 1 + block] for i in ix])
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    losses.append(loss.item())
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Bigram training loss")
plt.show()
print("first loss:", round(losses[0], 3), "last loss:", round(losses[-1], 3))

The loss drops: the model is learning which character tends to follow which.
Sample from it and you get character soup with the right letter frequencies,
the ceiling of what a bigram can do. The rest of the course lifts that ceiling.

In [ ]:
idx = torch.zeros((1, 1), dtype=torch.long)
for _ in range(120):
    logits, _ = model(idx[:, -1:])
    probs = F.softmax(logits[:, -1, :], dim=-1)
    idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
print(tok.decode(idx[0].tolist()))

In [ ]:
# The optimizer actually reduced the loss.
assert sum(losses[-5:]) / 5 < losses[0]